In [3]:
# Imports and constants
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

DATA_DIR = Path("a:/Software Projects/Delhi-AQI-Model/data")
RESULTS_DIR = Path("a:/Software Projects/Delhi-AQI-Model/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)


Data dir: a:\Software Projects\Delhi-AQI-Model\data
Results dir: a:\Software Projects\Delhi-AQI-Model\results


# Combined dataset generation

This notebook aggregates and merges daily data for Delhi AQI, Delhi humidity/temperature, Punjab winds, and Punjab fires for 2020-01-01 through 2024-12-31.

Approach:
- Read large hourly files in chunks and compute daily aggregates (mean/ sum/ counts)
- Align to common daily dates and merge
- Save merged CSV to `results/combined_2020_2024_daily.csv`


In [2]:
# Aggregate Delhi AQI to daily (handles large hourly file in chunks)

def aggregate_aqi_hourly(fp, date_col='event_timestamp', aqi_col='aqi'):
    sums = {}
    counts = {}
    parse_dates = [date_col]
    usecols = [date_col, aqi_col]
    for chunk in pd.read_csv(fp, usecols=lambda c: c in usecols, parse_dates=parse_dates, infer_datetime_format=True, chunksize=200_000):
        # drop missing aqi
        chunk = chunk.dropna(subset=[aqi_col])
        chunk['date'] = chunk[date_col].dt.strftime('%Y-%m-%d')
        gb = chunk.groupby('date')[aqi_col].agg(['sum','count'])
        for d, row in gb.iterrows():
            sums[d] = sums.get(d,0) + row['sum']
            counts[d] = counts.get(d,0) + row['count']
    # construct dataframe
    dates = sorted(sums.keys())
    df = pd.DataFrame({ 'date': pd.to_datetime(dates), 'avg_aqi': [sums[d]/counts[d] for d in dates], 'aqi_count': [counts[d] for d in dates] })
    df = df.set_index('date').sort_index()
    return df

# Try to load hourly AQI and aggregate
hourly_aqi_fp = DATA_DIR / 'delhi_air_quality_2024.csv'
print('Aggregating hourly AQI (this may take a few minutes)')
try:
    df_aqi_hourly = aggregate_aqi_hourly(hourly_aqi_fp)
    print('Hourly AQI aggregated:', df_aqi_hourly.shape)
except Exception as e:
    print('Hourly AQI aggregation failed:', e)
    df_aqi_hourly = pd.DataFrame()

# Also load daily recomputed and legacy AQI files and combine
def load_daily_aqi_from_daymonthyear(fp):
    df = pd.read_csv(fp)
    if {'Date','Month','Year'}.issubset(df.columns):
        df['date'] = pd.to_datetime(df['Year'].astype(str) + '-' + df['Month'].astype(str) + '-' + df['Date'].astype(str), dayfirst=False)
        df = df.set_index('date').sort_index()
        # prefer column name 'AQI' if present
        if 'AQI' in df.columns:
            out = df[['AQI']].rename(columns={'AQI':'avg_aqi'})
            return out
    return pd.DataFrame()

recomputed_fp = DATA_DIR / 'delhi_air_quality_2024_recomputed.csv'
legacy_fp = DATA_DIR / 'delhi_aqi_new.csv'

frames = []
for fp in [recomputed_fp, legacy_fp]:
    try:
        d = load_daily_aqi_from_daymonthyear(fp)
        if not d.empty:
            frames.append(d)
            print('Loaded daily AQI from', fp.name, 'rows:', len(d))
    except Exception as e:
        print('Failed loading', fp, e)

# Combine sources: prefer daily recomputed/legacy over hourly aggregates for overlapping days
if not df_aqi_hourly.empty:
    df_aqi = df_aqi_hourly.copy()
else:
    df_aqi = pd.DataFrame()

if frames:
    df_daily_sources = pd.concat(frames).sort_index()
    # if df_aqi has values, fill missing days with daily source; else use daily source
    if not df_aqi.empty:
        df_combined_aqi = df_aqi.combine_first(df_daily_sources[['avg_aqi']])
    else:
        df_combined_aqi = df_daily_sources[['avg_aqi']]
else:
    df_combined_aqi = df_aqi[['avg_aqi']].copy()

print('Combined AQI daily rows:', df_combined_aqi.shape)


Aggregating hourly AQI (this may take a few minutes)


C:\Users\Atharva Taras\AppData\Local\Temp\ipykernel_23088\798831541.py:8: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  for chunk in pd.read_csv(fp, usecols=lambda c: c in usecols, parse_dates=parse_dates, infer_datetime_format=True, chunksize=200_000):


Hourly AQI aggregation failed: unsupported operand type(s) for +: 'int' and 'str'
Loaded daily AQI from delhi_air_quality_2024_recomputed.csv rows: 366
Loaded daily AQI from delhi_aqi_new.csv rows: 1461
Combined AQI daily rows: (1827, 1)


In [3]:
# Load Delhi humidity/temperature (daily)
df_hum = pd.read_csv(DATA_DIR / 'delhi_humidity.csv', parse_dates=['date']).set_index('date').sort_index()
# rename columns for clarity
rename_map = {'meantemp':'mean_temp','humidity':'humidity','wind_speed':'delhi_wind_kmh','meanpressure':'mean_pressure'}
df_hum = df_hum.rename(columns=rename_map)
print('Delhi humidity rows:', df_hum.shape)


Delhi humidity rows: (1462, 4)


In [4]:
# Aggregate Punjab hourly wind to daily means
wind_fp = DATA_DIR / 'wind_speed_tel_hr_punjab_sw_pb_1970_2025.csv'
wind_date_col = 'Data Acquisition Time'
wind_val_col = 'Telemetry Hourly Wind Speed (Km/Hr)'

def aggregate_wind_hourly(fp, date_col=wind_date_col, val_col=wind_val_col, dt_format='%d-%m-%Y %H:%M'):
    sums = {}
    counts = {}
    std_acc = {}
    station_counts = {}
    for chunk in pd.read_csv(fp, usecols=[date_col, val_col], parse_dates=[date_col], dayfirst=True, chunksize=200_000, infer_datetime_format=True):
        chunk = chunk.dropna(subset=[val_col])
        # try parse if not parsed
        if not np.issubdtype(chunk[date_col].dtype, np.datetime64):
            chunk[date_col] = pd.to_datetime(chunk[date_col], format=dt_format, errors='coerce')
        chunk = chunk.dropna(subset=[date_col])
        chunk['date'] = chunk[date_col].dt.date
        gb = chunk.groupby('date')[val_col].agg(['sum','count','mean','std'])
        for d, row in gb.iterrows():
            dstr = d.strftime('%Y-%m-%d')
            sums[dstr] = sums.get(dstr,0) + row['sum']
            counts[dstr] = counts.get(dstr,0) + row['count']
            # accumulate for std later (we'll approximate by merging means weighted by counts)
            std_acc[dstr] = std_acc.get(dstr,[]) + [ (row['mean'], row['count'], row['std'] if not np.isnan(row['std']) else 0) ]
    # compute final mean and approximate std
    dates = sorted(sums.keys())
    means = [sums[d]/counts[d] for d in dates]
    stds = []
    for d in dates:
        parts = std_acc.get(d,[])
        # combine using weighted within-chunk variance approx (not exact but ok)
        # create approx combined variance from chunk means and stds
        total_n = sum(n for (_,n,_) in parts)
        if total_n <=1:
            stds.append(np.nan)
            continue
        # combine: sum of squares
        ss = 0.0
        for m,n,s in parts:
            # estimate sumsq = n*(s^2 + m^2)
            ss += n*((s**2) + (m**2))
        mean = sums[d]/counts[d]
        var = ss/total_n - mean**2
        stds.append(np.sqrt(var) if var>0 else 0.0)

    dfw = pd.DataFrame({'date':pd.to_datetime(dates),'mean_wind_kmh':means,'std_wind_kmh':stds})
    dfw = dfw.set_index('date').sort_index()
    return dfw

print('Aggregating hourly Punjab wind (this may take some time)')
try:
    df_wind = aggregate_wind_hourly(wind_fp)
    print('Wind aggregated rows:', df_wind.shape)
except Exception as e:
    print('Wind aggregation failed:', e)
    df_wind = pd.DataFrame()


Aggregating hourly Punjab wind (this may take some time)


C:\Users\Atharva Taras\AppData\Local\Temp\ipykernel_23088\2418680530.py:11: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  for chunk in pd.read_csv(fp, usecols=[date_col, val_col], parse_dates=[date_col], dayfirst=True, chunksize=200_000, infer_datetime_format=True):


Wind aggregated rows: (1183, 2)


In [5]:
# Aggregate Punjab fire events to daily counts and total FRP
fire_fp = DATA_DIR / 'punjab_fire_2020-2024.csv'

def inspect_csv_columns(fp, nrows=200):
    return pd.read_csv(fp, nrows=nrows).columns.tolist()

print('Inspecting fire file columns (sample) ...')
try:
    cols = inspect_csv_columns(fire_fp)
    print('Sample columns:', cols[:20])
except Exception as e:
    print('Failed to inspect fire file:', e)
    cols = []

# heuristics for date and frp columns
possible_date_cols = [c for c in cols if 'date' in c.lower() or 'time' in c.lower()]
possible_frp_cols = [c for c in cols if 'frp' in c.lower() or 'power' in c.lower()]
print('Detected date cols:', possible_date_cols)
print('Detected frp cols:', possible_frp_cols)

# fallback names
date_col = possible_date_cols[0] if possible_date_cols else None
frp_col = possible_frp_cols[0] if possible_frp_cols else None

# chunked aggregation
fire_sums = {}
fire_counts = {}
if date_col is None:
    print('No date-like column found for fires — aborting fire aggregation')
else:
    for chunk in pd.read_csv(fire_fp, usecols=lambda c: (c==date_col) or (frp_col and c==frp_col), parse_dates=[date_col], infer_datetime_format=True, chunksize=200_000, low_memory=True):
        # parse date only
        chunk = chunk.dropna(subset=[date_col])
        chunk['date'] = pd.to_datetime(chunk[date_col]).dt.date
        if frp_col and frp_col in chunk.columns:
            # convert frp to numeric
            chunk[frp_col] = pd.to_numeric(chunk[frp_col], errors='coerce').fillna(0)
            gb = chunk.groupby('date').agg({frp_col:['sum','count']})
            gb.columns = ['total_frp','fire_count']
            for d,row in gb.iterrows():
                dstr = d.strftime('%Y-%m-%d')
                fire_sums[dstr] = fire_sums.get(dstr,0) + row['total_frp']
                fire_counts[dstr] = fire_counts.get(dstr,0) + int(row['fire_count'])
        else:
            # just count events per day
            gb = chunk.groupby('date').size()
            for d,n in gb.items():
                dstr = d.strftime('%Y-%m-%d')
                fire_counts[dstr] = fire_counts.get(dstr,0) + int(n)

# assemble df
dates = sorted(set(list(fire_counts.keys()) + list(fire_sums.keys())))
if dates:
    df_fire = pd.DataFrame({'date':pd.to_datetime(dates), 'fire_count':[fire_counts.get(d,0) for d in dates], 'total_frp':[fire_sums.get(d,0) for d in dates]})
    df_fire = df_fire.set_index('date').sort_index()
    print('Fire aggregated rows:', df_fire.shape)
else:
    df_fire = pd.DataFrame()
    print('No fire aggregation results')


Inspecting fire file columns (sample) ...
Sample columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
Detected date cols: ['acq_date', 'acq_time']
Detected frp cols: ['frp']


C:\Users\Atharva Taras\AppData\Local\Temp\ipykernel_23088\2792307104.py:31: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  for chunk in pd.read_csv(fire_fp, usecols=lambda c: (c==date_col) or (frp_col and c==frp_col), parse_dates=[date_col], infer_datetime_format=True, chunksize=200_000, low_memory=True):


Fire aggregated rows: (1822, 2)


In [6]:
# Merge all daily series into final dataset covering 2020-01-01 to 2024-12-31
start = pd.Timestamp('2020-01-01')
end = pd.Timestamp('2024-12-31')
dates = pd.date_range(start, end, freq='D')
final = pd.DataFrame(index=dates)

# helper to align and merge
if 'df_combined_aqi' in globals() and not df_combined_aqi.empty:
    final = final.join(df_combined_aqi[['avg_aqi']])
else:
    print('Warning: no AQI data available to join')

if 'df_hum' in globals() and not df_hum.empty:
    final = final.join(df_hum[['mean_temp','humidity']])
else:
    print('Warning: no humidity data available to join')

if 'df_wind' in globals() and not df_wind.empty:
    final = final.join(df_wind[['mean_wind_kmh','std_wind_kmh']])
else:
    print('Warning: no wind data available to join')

if 'df_fire' in globals() and not df_fire.empty:
    final = final.join(df_fire[['fire_count','total_frp']])
else:
    print('Warning: no fire data available to join')

# basic checks
print('Final shape:', final.shape)
print('Date range:', final.index.min(), final.index.max())
print('Missing per column:')
print(final.isna().mean())

# Save
out_fp = RESULTS_DIR / 'combined_2020_2024_daily.csv'
final.to_csv(out_fp, index_label='date')
print('Saved merged dataset to', out_fp)

# show a few rows
final.head()


Final shape: (2193, 7)
Date range: 2020-01-01 00:00:00 2024-12-31 00:00:00
Missing per column:
avg_aqi          0.166895
mean_temp        1.000000
humidity         1.000000
mean_wind_kmh    0.379389
std_wind_kmh     0.380301
fire_count       0.003192
total_frp        0.003192
dtype: float64
Saved merged dataset to a:\Software Projects\Delhi-AQI-Model\results\combined_2020_2024_daily.csv


,avg_aqi,mean_temp,humidity,mean_wind_kmh,std_wind_kmh,fire_count,total_frp
2020-01-01,NaN,NaN,NaN,NaN,NaN,486.0,1569.68
2020-01-02,NaN,NaN,NaN,NaN,NaN,319.0,1003.92
2020-01-03,NaN,NaN,NaN,NaN,NaN,299.0,837.23
2020-01-04,NaN,NaN,NaN,NaN,NaN,361.0,1060.21
2020-01-05,NaN,NaN,NaN,NaN,NaN,698.0,1964.12


## Notes and assumptions

- Hourly datasets (Delhi AQI hourly, Punjab wind) are aggregated to daily by taking the mean (for continuous variables) and counts/sums for event data.
- Punjab fire data is aggregated to daily `fire_count` and `total_frp` (sum of FRP when available). If FRP is unavailable, `total_frp` will be zero and only counts will be used.
- Missing values are left as NaN so you can decide how to impute or filter them.
- The final CSV is saved as `results/combined_2020_2024_daily.csv` with a daily index from 2020-01-01 to 2024-12-31.

Next steps (optional):
- Add station-level aggregates or spatial joins if you want per-station features.
- Impute missing values (e.g., forward-fill or seasonal interpolation).

✅ If you'd like, I can run validation checks (missingness by year, sample plots) or tune the aggregation choices (median vs mean, max wind, etc.).

In [18]:
final.isnull().sum()

avg_aqi           366
mean_temp        2193
humidity         2193
mean_wind_kmh     832
std_wind_kmh      834
fire_count          7
total_frp           7
dtype: int64

In [4]:
final = pd.read_csv(r'a:\Software Projects\Delhi-AQI-Model\results\combined_2020_2024_daily.csv')
final.head()

,date,avg_aqi,mean_temp,humidity,mean_wind_kmh,std_wind_kmh,fire_count,total_frp
0,2020-01-01,NaN,NaN,NaN,NaN,NaN,486.0,1569.68
1,2020-01-02,NaN,NaN,NaN,NaN,NaN,319.0,1003.92
2,2020-01-03,NaN,NaN,NaN,NaN,NaN,299.0,837.23
3,2020-01-04,NaN,NaN,NaN,NaN,NaN,361.0,1060.21
4,2020-01-05,NaN,NaN,NaN,NaN,NaN,698.0,1964.12


In [23]:
new = final.dropna(subset=['avg_aqi'])

In [25]:
new.isnull().sum()

avg_aqi          0
mean_temp        0
humidity         0
mean_wind_kmh    0
std_wind_kmh     0
fire_count       0
total_frp        0
dtype: int64

In [26]:
new.to_csv(r'results\combined_clean_data.csv', index=False)

In [8]:
# Modeling imports and helper functions
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import os

# helper
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def save_fig(fig, name):
    path = RESULTS_DIR / name
    fig.savefig(path, bbox_inches='tight', dpi=150)
    print('Saved', path)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'models').mkdir(parents=True, exist_ok=True)


# Modeling plan
# - Load cleaned combined dataset
# - Create date features and lag features for `avg_aqi`
# - Train multiple regression models (Linear, Ridge, RandomForest, Gradient Boosting) using time-aware split
# - Evaluate on 2024 holdout set and save metrics and plots (predicted vs actual, residuals, feature importances)


In [9]:
# Load cleaned dataset and run basic EDA
# Prefer the full combined dataset with dates (if available) for modeling
try:
    df = pd.read_csv(RESULTS_DIR / 'combined_2020_2024_daily.csv', parse_dates=['date'])
    df = df.set_index('date').sort_index()
    print('Loaded full combined dataset rows:', df.shape)
except Exception:
    # fallback to cleaned file (may not contain dates)
    df = pd.read_csv(RESULTS_DIR / 'combined_clean_data.csv')
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date').sort_index()
    else:
        start = pd.Timestamp('2020-01-01')
        df.index = pd.date_range(start, periods=len(df), freq='D')
    print('Loaded fallback cleaned dataset rows:', df.shape)

# Basic info
if 'avg_aqi' not in df.columns or df['avg_aqi'].dropna().empty:
    print('Warning: `avg_aqi` has no valid values in the loaded dataset. You may want to check source files.')

print(df[['avg_aqi']].describe())

# missingness
miss = df.isna().mean()
print('\nMissing fraction per column:')
print(miss)

# plot timeseries of AQI
if not df['avg_aqi'].dropna().empty:
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(df.index, df['avg_aqi'], label='avg_aqi')
    ax.set_title('Time series of avg_aqi')
    ax.set_ylabel('AQI')
    ax.legend()
    save_fig(fig, 'aqi_timeseries.png')
    plt.close(fig)

    # histogram
    fig = plt.figure(figsize=(6,4))
    sns.histplot(df['avg_aqi'].dropna(), bins=40, kde=True)
    plt.title('Distribution of avg_aqi')
    save_fig(fig, 'aqi_distribution.png')
    plt.close(fig)
else:
    print('No non-null avg_aqi values to plot')

Loaded full combined dataset rows: (2193, 7)
           avg_aqi
count  1827.000000
mean    200.856596
std     106.168227
min      19.000000
25%     110.000000
50%     187.000000
75%     279.000000
max     500.000000

Missing fraction per column:
avg_aqi          0.166895
mean_temp        1.000000
humidity         1.000000
mean_wind_kmh    0.379389
std_wind_kmh     0.380301
fire_count       0.003192
total_frp        0.003192
dtype: float64
Saved a:\Software Projects\Delhi-AQI-Model\results\aqi_timeseries.png
Saved a:\Software Projects\Delhi-AQI-Model\results\aqi_distribution.png


In [5]:
# Inspect combined_clean_data.csv header
with open(RESULTS_DIR / 'combined_clean_data.csv', 'r', encoding='utf-8') as f:
    header = f.readline().strip()
print('Header:', header)
pd.read_csv(RESULTS_DIR / 'combined_clean_data.csv', nrows=3).head()

Header: avg_aqi,mean_temp,humidity,mean_wind_kmh,std_wind_kmh,fire_count,total_frp


,avg_aqi,mean_temp,humidity,mean_wind_kmh,std_wind_kmh,fire_count,total_frp


In [11]:
# Feature engineering: date features and lags
feat = df.copy()
# date features
feat['month'] = feat.index.month
feat['dayofyear'] = feat.index.dayofyear
feat['weekday'] = feat.index.weekday

# lag features for avg_aqi
for lag in [1,2,3,7]:
    feat[f'aqi_lag_{lag}'] = feat['avg_aqi'].shift(lag)

# rolling mean
feat['aqi_roll7_mean'] = feat['avg_aqi'].rolling(7, min_periods=1).mean().shift(1)

# drop rows where target is missing
feat = feat.dropna(subset=['avg_aqi'])
# DO NOT drop rows with feature NaNs here; we'll impute in preprocessing pipeline

print('After feature engineering, rows:', feat.shape)

# candidate features
candidate_features = [c for c in feat.columns if c!='avg_aqi']
print('Candidate feature count:', len(candidate_features))
print(candidate_features)

After feature engineering, rows: (1827, 15)
Candidate feature count: 14
['mean_temp', 'humidity', 'mean_wind_kmh', 'std_wind_kmh', 'fire_count', 'total_frp', 'month', 'dayofyear', 'weekday', 'aqi_lag_1', 'aqi_lag_2', 'aqi_lag_3', 'aqi_lag_7', 'aqi_roll7_mean']


In [12]:
# Time-aware train/test split
train_end = pd.Timestamp('2023-12-31')
val_start = pd.Timestamp('2024-01-01')

train = feat.loc[:train_end].copy()
test = feat.loc[val_start:].copy()

print('Train rows:', train.shape)
print('Test rows:', test.shape)

X_train = train[candidate_features]
y_train = train['avg_aqi']
X_test = test[candidate_features]
y_test = test['avg_aqi']

# Preprocessing pipeline for numeric features
numeric_features = X_train.columns.tolist()
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features)
])

# evaluation splits
tscv = TimeSeriesSplit(n_splits=5)


Train rows: (1095, 15)
Test rows: (732, 15)


In [13]:
# Models to evaluate
models = {}
# simple linear models
models['LinearRegression'] = Pipeline(steps=[('pre', preprocessor), ('model', LinearRegression())])
models['Ridge'] = Pipeline(steps=[('pre', preprocessor), ('model', Ridge())])

# tree-based models
models['RandomForest'] = Pipeline(steps=[('pre', preprocessor), ('model', RandomForestRegressor(random_state=42, n_jobs=-1))])
models['HistGB'] = Pipeline(steps=[('pre', preprocessor), ('model', HistGradientBoostingRegressor(random_state=42))])

# try XGBoost if available
try:
    import xgboost as xgb
    models['XGBoost'] = Pipeline(steps=[('pre', preprocessor), ('model', xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1))])
    xgb_installed = True
except Exception:
    xgb_installed = False
    print('xgboost not available; skipping')

# Hyperparameter search spaces
param_distributions = {
    'RandomForest': {
        'model__n_estimators': [100, 200, 400],
        'model__max_depth': [5,10,20,None],
        'model__min_samples_split': [2,5,10]
    },
    'HistGB': {
        'model__max_iter': [100,200,400],
        'model__max_depth': [3,7,12],
        'model__learning_rate': [0.01, 0.05, 0.1]
    }
}
if xgb_installed:
    param_distributions['XGBoost'] = {
        'model__n_estimators': [100,200,400],
        'model__max_depth': [3,6,9],
        'model__learning_rate': [0.01,0.05,0.1]
    }

# Storage for results
metrics = []
trained_models = {}

# Function to fit, optionally search hyperparams, and evaluate
from datetime import timedelta

def fit_and_evaluate(name, pipeline, param_dist=None, n_iter=20):
    print('\nTraining', name)
    if param_dist:
        r = RandomizedSearchCV(pipeline, param_dist, n_iter=min(n_iter, 10), cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1, random_state=42)
        r.fit(X_train, y_train)
        best = r.best_estimator_
        print('Best params:', r.best_params_)
        trained = best
    else:
        pipeline.fit(X_train, y_train)
        trained = pipeline

    # predictions on test
    y_pred = trained.predict(X_test)
    r_rmse = rmse(y_test, y_pred)
    r_mae = mean_absolute_error(y_test, y_pred)
    r_r2 = r2_score(y_test, y_pred)
    print(f'{name} Test RMSE: {r_rmse:.3f}  MAE: {r_mae:.3f}  R2: {r_r2:.3f}')

    # save model
    joblib.dump(trained, RESULTS_DIR / 'models' / f'{name}.joblib')
    print('Saved model to', RESULTS_DIR / 'models' / f'{name}.joblib')

    # plots
    # time series plot
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(y_test.index, y_test.values, label='actual')
    ax.plot(y_test.index, y_pred, label='predicted')
    ax.set_title(f'{name}: Actual vs Predicted (test)')
    ax.legend()
    save_fig(fig, f'{name}_actual_vs_predicted.png')
    plt.close(fig)

    # scatter
    fig = plt.figure(figsize=(6,6))
    sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title(f'{name}: Predicted vs Actual')
    save_fig(fig, f'{name}_pred_vs_actual_scatter.png')
    plt.close(fig)

    # residuals
    resid = y_test - y_pred
    fig = plt.figure(figsize=(6,4))
    sns.histplot(resid, bins=40, kde=True)
    plt.title(f'{name}: Residuals (test)')
    save_fig(fig, f'{name}_residuals.png')
    plt.close(fig)

    # feature importance if available
    try:
        model_obj = trained.named_steps['model']
        if hasattr(model_obj, 'feature_importances_'):
            fi = model_obj.feature_importances_
            # map back through preprocessor to original feature order (numeric_features)
            fig = plt.figure(figsize=(8,6))
            idx = np.argsort(fi)[::-1][:25]
            sns.barplot(x=fi[idx], y=np.array(numeric_features)[idx])
            plt.title(f'{name}: Feature importances')
            save_fig(fig, f'{name}_feature_importances.png')
            plt.close(fig)
        elif hasattr(model_obj, 'coef_'):
            coef = model_obj.coef_
            fig = plt.figure(figsize=(8,6))
            idx = np.argsort(np.abs(coef))[::-1][:25]
            sns.barplot(x=coef[idx], y=np.array(numeric_features)[idx])
            plt.title(f'{name}: Coefficients')
            save_fig(fig, f'{name}_coefficients.png')
            plt.close(fig)
    except Exception as e:
        print('Could not extract feature importances for', name, e)

    metrics.append({'model':name, 'rmse':r_rmse, 'mae':r_mae, 'r2':r_r2})
    trained_models[name] = trained

# Run for models
for name, pipe in models.items():
    pd.options.display.max_colwidth = 200
    param_dist = param_distributions.get(name)
    fit_and_evaluate(name, pipe, param_dist)

# save metrics
metrics_df = pd.DataFrame(metrics).sort_values('rmse')
metrics_df.to_csv(RESULTS_DIR / 'aqi_model_metrics.csv', index=False)
print('\nSaved metrics to', RESULTS_DIR / 'aqi_model_metrics.csv')


xgboost not available; skipping

Training LinearRegression
LinearRegression Test RMSE: 35.992  MAE: 24.602  R2: 0.868
Saved model to a:\Software Projects\Delhi-AQI-Model\results\models\LinearRegression.joblib


a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['mean_temp' 'humidity']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['mean_temp' 'humidity']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Saved a:\Software Projects\Delhi-AQI-Model\results\LinearRegression_actual_vs_predicted.png
Saved a:\Software Projects\Delhi-AQI-Model\results\LinearRegression_pred_vs_actual_scatter.png
Saved a:\Software Projects\Delhi-AQI-Model\results\LinearRegression_residuals.png
Saved a:\Software Projects\Delhi-AQI-Model\results\LinearRegression_coefficients.png

Training Ridge
Ridge Test RMSE: 36.015  MAE: 24.642  R2: 0.868
Saved model to a:\Software Projects\Delhi-AQI-Model\results\models\Ridge.joblib
Saved a:\Software Projects\Delhi-AQI-Model\results\Ridge_actual_vs_predicted.png


a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['mean_temp' 'humidity']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['mean_temp' 'humidity']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Saved a:\Software Projects\Delhi-AQI-Model\results\Ridge_pred_vs_actual_scatter.png
Saved a:\Software Projects\Delhi-AQI-Model\results\Ridge_residuals.png
Saved a:\Software Projects\Delhi-AQI-Model\results\Ridge_coefficients.png

Training RandomForest


...

Saved a:\Software Projects\Delhi-AQI-Model\results\RandomForest_actual_vs_predicted.png
Saved a:\Software Projects\Delhi-AQI-Model\results\RandomForest_pred_vs_actual_scatter.png
Saved a:\Software Projects\Delhi-AQI-Model\results\RandomForest_residuals.png
Saved a:\Software Projects\Delhi-AQI-Model\results\RandomForest_feature_importances.png

Training HistGB


a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['mean_temp' 'humidity']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Best params: {'model__max_iter': 100, 'model__max_depth': 3, 'model__learning_rate': 0.05}
HistGB Test RMSE: 37.844  MAE: 27.518  R2: 0.854
Saved model to a:\Software Projects\Delhi-AQI-Model\results\models\HistGB.joblib
Saved a:\Software Projects\Delhi-AQI-Model\results\HistGB_actual_vs_predicted.png


a:\Software Projects\Delhi-AQI-Model\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['mean_temp' 'humidity']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Saved a:\Software Projects\Delhi-AQI-Model\results\HistGB_pred_vs_actual_scatter.png
Saved a:\Software Projects\Delhi-AQI-Model\results\HistGB_residuals.png

Saved metrics to a:\Software Projects\Delhi-AQI-Model\results\aqi_model_metrics.csv


In [14]:
# Summary and pick best model
metrics_df = pd.read_csv(RESULTS_DIR / 'aqi_model_metrics.csv')
metrics_df

best = metrics_df.sort_values('rmse').iloc[0]
print('Best model:', best['model'], 'RMSE:', best['rmse'])

# write markdown report
md = f"""# AQI Model Metrics Summary

Best model: **{best['model']}**

| model | rmse | mae | r2 |
|---|---:|---:|---:|
"""
for _,r in metrics_df.iterrows():
    md += f"| {r['model']} | {r['rmse']:.3f} | {r['mae']:.3f} | {r['r2']:.3f} |\n"

(RESULTS_DIR / 'aqi_model_metrics.md').write_text(md)
print('Saved markdown summary to', RESULTS_DIR / 'aqi_model_metrics.md')


Best model: LinearRegression RMSE: 35.99169029230793
Saved markdown summary to a:\Software Projects\Delhi-AQI-Model\results\aqi_model_metrics.md
